In [21]:
import json
import time
from IPython.display import display, Markdown, HTML
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

MODEL = "llama3.2:3b"

#Initialize the model
llm = ChatOllama(model=MODEL)

In [ ]:
def build_messages(message_dicts):
    type_map = {
        "system": SystemMessage,
        "user": HumanMessage,
        "assistant" : AIMessage
    }
    messages = []
    for m in message_dicts:
        role = m["role"]
        message_class = type_map[role]
        message = message_class(content=m["content"])
        messages.append(message)
    return messages  # or

    # return [type_map[m["role"]](content=m["content"]) for m in message_dicts]

def chat(messages, model=None):
    _llm = ChatOllama(model=MODEL) if model else llm
    lc_messages = build_messages(messages)
    start = time.time()
    response = _llm.invoke(lc_messages)
    elapsed = time.time() - start
    content = response.content
    display(Markdown(content))
    print(f"\n⏱️ Response time {elapsed:.2f}s")
    return content

def show_messages(messages):
    colors = {"system": "#e74c3c", "user": "#3498db", "assistant": "#2ecc71"}
    html = ""

    for msg in messages:
        role = msg["role"]
        color = colors.get(role, "#888")
        html += (
            f'<div style="margin:6px 0;padding:8px 12px;border-left:4px solid {color};'
            f'background:#1e1e1e;border-radius:4px;">'
            f'<strong style="color:{color};text-transform:uppercase;">{role}</strong>'
            f'<br><span style="color:#ccc;">{msg["content"]}</span></div>'
        )
    display(HTML(html))

print(f"✅ Using model: {MODEL}")

✅ Using model: llama3.2:3b


In [40]:
user_question = "Explain about cricket."

system_prompts = [
    {"label":'Pirate', "prompt":"You are a pirate captain. Respond to everything in pirate speak with nautical metaphors."},
    {
    "label": "Coach",
    "prompt": "You are a cricket coach. Respond with cricketing knowledge. Be concise and accurate."
    },
    {
    "label": "Poet",
    "prompt": "You are a poet. Respond to everything in the form of a short rhyming poem."
    }
]

for sp in system_prompts:
    print(f"PERSONA : {sp['label']}")
    messages = [
        {"role":"system", "content": sp["prompt"]},
        {"role":"user", "content":user_question}
    ]
    show_messages(messages)
    _ = chat(messages)

PERSONA : Pirate


Ye be wantin' to know about this "cricket" business, eh? Alright then, listen close and I'll tell ye the tale of the game.

Cricket be a sport, like sailin' through treacherous waters – ye gotta navigate by feelin', intuition, and skill. The objective be to score more runs than the other ship (team) by bat-tin' and runnin'.

The game be divided into innings, like segments o' a charted course. Each team gets a chance to sail their ship (bat) and plunder the enemy's treasure (runs). A batsman's role be similar to that o' a lookout: ye gotta keep watch for the opponent's crew (pitcher) and anticipate their moves.

There be different types o' ships, like bowlers, fast bowlers, swing bowlers – each with its own strengths and weaknesses. And then there be the wickets, like the anchors that hold the ship steady. If ye get out, ye're lost at sea!

Now, I know what ye be thinkin', "Pirate captain, why do they have all these rules?" Well, matey, it's like navigatin' through a maze – ye gotta follow the chart and avoid the rocks o' disaster.

Cricket be a complex game, but with patience and practice, ye can become a master navigator of the pitch. So hoist the sails and set course for adventure!


⏱️ Response time 21.46s
PERSONA : Coach


Cricket is a team sport played between two teams, each consisting of 11 players, with the objective of scoring runs by hitting a ball bowled by the opposing team's bowler.

The game is divided into innings, with each team getting a chance to bat and bowl. The batting team sends two batsmen onto the field, who take turns to hit the ball, while the bowling team tries to get them out.

Here are some key aspects of cricket:

**Key Positions:**

* Bowler (Bowls the ball)
* Batsman (Hhits the ball)
* Wicketkeeper (Stands behind wickets and catches balls)

**Types of Shots:**

* Straight drive
* Cut
* Pull
* Cover drive

**Ways to Get Out:**

* Bowled (hit by the ball, knocking over wickets)
* Lbw (leg before wicket, where the umpire thinks the ball would have hit the wickets if it hadn't been blocked)
* Caught (hitting the ball, which is then caught by a fielder)
* Run out (hit while running, and fielder touches the wicket)

**Basic Strategies:**

* Build a strong batting partnership
* Take regular wickets to limit opposition runs
* Use various bowling techniques to keep batsmen guessing

That's a brief overview of cricket! Do you have any specific questions or topics you'd like me to expand on?


⏱️ Response time 11.00s
PERSONA : Poet


A sport so fine, with bat and ball,
Cricket's the game that stands tall.
Two teams play on, with skill and might,
Each trying to score, or take out the light.

The batsman hits, with a mighty swing,
While bowlers cast, with a spinning ring.
Wickets fall down, with a sorrowful sound,
As runs are scored, and the team's renown.

Overs pass by, like leaves in the air,
As fielders run, with a speedy care.
The crowd cheers on, with a joyful shout,
For cricket's magic, that's always about.


⏱️ Response time 4.11s


In [25]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a data extraction assistant. "
            "Always respond with valid JSON only — no extra text. "
            "Extract: name, topic, and key_points (as a list)."
        )
    },
    {
        "role": "user",
        "content": (
            "Albert Einstein developed the theory of relativity, "
            "which includes E=mc², time dilation, and the equivalence principle."
        )
    }
]

show_messages(messages)
response_text = chat(messages)

# Try to parse the JSON response
try:
    parsed = json.loads(response_text)
    print("\n✅ Valid JSON! Parsed result:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError :
    print("\n⚠️ Response was not valid JSON — this shows the challenge of format control!")

{"name": "Albert Einstein", "topic": "Theory of Relativity", "key_points": ["E=mc²", "time dilation", "equivalence principle"]}


⏱️ Response time: 2.62s

✅ Valid JSON! Parsed result:
{
  "name": "Albert Einstein",
  "topic": "Theory of Relativity",
  "key_points": [
    "E=mc\u00b2",
    "time dilation",
    "equivalence principle"
  ]
}


---

## 2. The User Prompt — Driving the Conversation

The **user prompt** is the direct input from the human. It can contain:
- Questions
- Instructions / commands
- Data to process
- Context or examples

### Experiment 2A: Vague vs. Specific User Prompts

In [28]:
print("VAGUE PROMPT")
system = {"role": "system","content": "You are a helpful assistant. Keep responses concise."}

vague = [
    system,
    {"role": "user","content": "Tell me about Python."} 
]
show_messages(vague)
_ = chat(vague)

VAGUE PROMPT


Python is a high-level, interpreted programming language that's widely used for various purposes:

* Web development (e.g., Django, Flask)
* Data analysis and science (e.g., NumPy, pandas, scikit-learn)
* Artificial intelligence and machine learning (e.g., TensorFlow, Keras)
* Automation and scripting
* Education and research

Python is known for its simplicity, readability, and flexibility, making it a popular choice among developers and non-technical users alike.


⏱️ Response time: 3.92s


In [29]:
print("=" * 60)
print("SPECIFIC PROMPT")
print("=" * 60)

specific_prompt = [
    system,
    {"role":'user', "content": (
        "List 3 advantages of AI,"
        "with one sentence each.Format as a numbered list"
    )}
]
show_messages(specific_prompt)
_ = chat(specific_prompt)

SPECIFIC PROMPT


Here are three advantages of AI:

1. AI can automate repetitive and mundane tasks, freeing up time for more creative and strategic work.
2. AI can analyze vast amounts of data quickly and accurately, providing valuable insights that human analysts might miss.
3. AI-powered chatbots and virtual assistants can provide 24/7 customer support and improve overall user experience.


⏱️ Response time: 3.42s


### Experiment 2B: User Prompt with Embedded Data

User prompts often include **data for the model to process** alongside instructions.

In [32]:
messages = [
    {"role": "system", "content": "You are sentiment analysis assistant.Classify each review and explain briefly."},
    {"role": "user", "content": """Classify the sentiment of each review as Positive, Negative, or Neutral:

Review 1: "This product is amazing! Best purchase I've ever made."
Review 2: "Terrible quality. Broke after one day. Never buying again."
Review 3: "It's okay. Does what it says, nothing special."
"""}
]
show_messages(messages)
_ = chat(messages)

Here are the classifications with brief explanations:

**Review 1: Positive**
The reviewer uses strong positive language ("amazing", "best purchase") and expresses satisfaction with their purchase.

**Review 2: Negative**
The reviewer uses strong negative language ("terrible quality", "broke after one day") and warns others not to buy the product, indicating a strong negative sentiment.

**Review 3: Neutral**
The reviewer is neutral in their tone, using words like "it's okay" which imply a lack of enthusiasm or excitement. They also state that the product does what it says, but doesn't express any strong positive or negative emotions.


⏱️ Response time: 10.40s


## 3. The Assistant Role — Pre-filling & Multi-turn Conversations

In [46]:
# Stimulating a multi turn conversation
messages = [
    {"role": "system", "content": "You are a math tutor."},
    {"role": "user", "content": "What is 15% of 200? "},
    {"role": "assistant", "content": "15% of 200 is 30. Here's how: 15/100 × 200 = 0.15 × 200 = 30."},
    {"role": "user", "content": "What is 15% of 300?"}
]

show_messages(messages)
_ = chat(messages)

To find 15% of 300, I'll multiply 300 by 0.15 (since 15% is equal to 15/100 or 0.15).

300 × 0.15 = 45.

So, 15% of 300 is 45.


⏱️ Response time 9.95s


### Experiment 3B: Building a Conversation Turn by Turn

Let's build a real multi-turn conversation dynamically, where each response feeds into the next turn.

In [55]:
conversation = [
    {"role": "system", "content": "You are a cricket analysis assitant. Keep each responses to 2-3 sentences"}
]
user_turns = [
    "A team need to score 15 runs in 6 ball with 2 wickets in hand. 1st ball batsman hits a six, the next ball the batsman was bowled. How many runs need for the team to win",
    "The next batsman enters and hits a six, remaining three balls takes three singles.",
    "Does the batting team win?"
]
for i, user_msg in enumerate(user_turns, 1):
    print(f"TURN{i}")

    conversation.append({"role": "user", "content": user_msg})
    show_messages(conversation)

    response = llm.invoke(build_messages(conversation))
    assistant_msg = response.content
    display(Markdown(assistant_msg))

    conversation.append({"role": "assistant", "content": assistant_msg})


TURN1


With 4 balls remaining and 2 wickets intact, the team still has a decent chance of winning. However, the loss of a wicket after hitting a six reduces their advantage slightly. They would now need to score at least 11-12 runs per ball to chase down the target.

TURN2


With just 3 balls remaining, the team's chances of winning are looking good after scoring 9 runs in those 3 deliveries. The batsmen still need to hit a boundary or score quickly off the last ball to seal the win. They would aim for a loose delivery to play an aggressive shot.

TURN3


Yes, with 3 singles in the final 3 balls, the batting team manages to reach the target of 15 runs and wins the game by virtue of the bowlers' inability to take any more wickets before the target was reached.

### Experiment 4B: Comparing System Prompt Strictness

How does the **level of detail** in a system prompt affect the model's behavior?

In [57]:
query = "Explain about car?"

system_levels = [
    {"label": "Minimal",
     "prompt": "You are a helpful assistant."},

    {"label": "Moderate",
     "prompt": "You are a sales person. Give advice in 5 bullet points"},

     {"label": "Strict",
      "prompt": "You are a car mechanic. "
      "Keys:"
      "Respond in three bullet points. "
      "Include a main function of a car."}
]

for sl in system_levels:
    print(f"SYSTEM LEVEL : {sl['label']}")

    messages = [
        {"role": "system", "content": sl["prompt"]},
        {"role": "user", "content": query}
    ]

    show_messages(messages)
    _ = chat(messages)

SYSTEM LEVEL : Minimal


A car, also known as an automobile, is a self-powered wheeled vehicle designed for transportation. Here's a comprehensive overview:

**Key Components:**

1. **Engine**: The engine is the heart of the car, responsible for converting fuel into energy to power the vehicle.
2. **Transmission**: The transmission system transmits the energy from the engine to the wheels, adjusting the speed and torque as needed.
3. **Drivetrain**: The drivetrain consists of gears, shafts, and axles that transmit power from the transmission to the wheels.
4. **Wheels and Tires**: The wheels and tires provide the necessary surface contact with the road, while also absorbing shocks and vibrations.

**How it Works:**

1. **Fuel Injection**: Fuel is injected into the engine's cylinders, where it is ignited by a spark plug, producing power.
2. **Power Transfer**: Power is transferred from the engine to the transmission, which adjusts the gear ratio based on speed and load.
3. **Drive Train**: The transmission sends power to the drivetrain, which transmits it to the wheels through the axles.
4. **Motion**: The wheels rotate, propelling the car forward.

**Types of Cars:**

1. **Sedan**: A passenger car with a fixed roof and a rear-engine layout.
2. **Hatchback**: A car with a rear door that swings upward to access the cargo area.
3. **SUV (Sport Utility Vehicle)**: A vehicle designed for off-road driving, typically featuring higher ground clearance and four-wheel drive.
4. **Truck**: A vehicle designed for hauling heavy loads or towing trailers.

**Safety Features:**

1. **Airbags**: Deployable bags that deploy in the event of a crash to cushion occupants.
2. **Anti-lock Braking System (ABS)**: A system that prevents wheels from locking up during hard braking, maintaining traction and stability.
3. **Electronic Stability Control (ESC)**: A system that helps the car stabilize during sudden maneuvers or changes in direction.

**Modern Car Technologies:**

1. **Electric Vehicles (EVs)**: Cars powered by electric motors, often with battery storage units.
2. **Hybridization**: Combining a conventional engine with an electric motor for improved fuel efficiency.
3. **Autonomous Driving**: Systems that enable cars to navigate and control themselves without human intervention.

This is just a brief overview of the world of cars! If you have any specific questions or topics you'd like me to expand on, feel free to ask!


⏱️ Response time 19.44s
SYSTEM LEVEL : Moderate


As a salesperson, I'd be happy to provide some general information and tips about cars. Here are five key things to consider:

• **Know Your Budget**: Before buying a car, determine how much you can afford to spend on a vehicle, including the purchase price, financing costs, insurance, fuel, maintenance, and repairs. Consider your income, savings, debts, and other expenses to ensure you have enough room in your budget for a new car.

• **Research Different Models**: Look into various car models, their features, performance, reliability, and safety ratings. Read reviews from reputable sources like Kelley Blue Book or Edmunds, and compare different models side by side to find the best fit for your needs.

• **Test Drive Cars**: Once you've narrowed down your options, take the cars for a test drive to get a feel for their handling, comfort, and features. Pay attention to any issues you experience during the drive, such as vibration or noise.

• **Check the Vehicle History**: Use services like Carfax or AutoCheck to get detailed reports on the car's ownership history, accidents, and any major repairs that may have been done. This can help you avoid purchasing a car with hidden problems.

• **Negotiate Wisely**: When it comes time to make an offer, don't be afraid to negotiate. Research the market value of the car using tools like Kelley Blue Book or Edmunds, and use this information as a basis for your offer. Don't fall for high-pressure sales tactics – stay calm and patient, and keep working towards a fair deal.

I hope these tips are helpful! Do you have any specific questions about cars?


⏱️ Response time 12.67s
SYSTEM LEVEL : Strict


Here are three key points about cars:

• **Main Function**: The primary function of a car is to provide transportation, allowing individuals and goods to move from one place to another.
• **Engine Power**: A car's engine is its heart, providing the power needed to propel the vehicle forward. Modern engines use a combination of air, fuel, and spark to generate energy.
• **Safety Features**: Most modern cars come equipped with advanced safety features such as airbags, anti-lock braking systems (ABS), and electronic stability control (ESC) to protect the occupants in the event of an accident or sudden stop.


⏱️ Response time 4.84s
